In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.window import Window

In [0]:
df=spark.read.parquet('/Volumes/workspace/default/projecttradevolume/1_min/INFY/2026/01/20/*')

In [0]:
display(df)

In [0]:
df=df.orderBy(col('DateTime').asc()).select(df.columns[:-1])
df_up=(
    df
    .withColumn('day_change',round(col('Close')-lag(col('Close'),1).over(Window.orderBy(col('DateTime'))),3))
    .withColumn('day_change%',round(((col('day_change')/lag(col('Close'),1).over(Window.orderBy(col('DateTime'))))*100),2))
    # .withColumn()
    .withColumn('h-l',round(
        (col('High')-col('Low'))
        ,2
        )
    )
    .withColumn('c-l',round(
        (col('Close')-col('Low'))
        ,2
        )
    )
    .withColumn('o-l',round(
        (col('Open')-col('Low'))
        ,2
        )
    )
    .withColumn('cl%',round(
        (col('c-l')/col('h-l'))*100
        ,2
        )
    )
    .withColumn('ol%',round(
        (col('o-l')/col('h-l'))*100
        ,2
        )
    )
    .withColumn('candle%',round(
        (col('cl%')-col('ol%'))
        ,2
        )
    )
    .withColumn(
        'CandlePatten'
        ,when(abs(col('candle%'))==100,lit('Marubozu'))
        .when(abs(col('candle%'))==0,lit('Doji'))
        .otherwise('NA')
    )

)
display(df_up)